# ReefScan-ID: NOAA bleaching dataset check

Check of the NOAA PIFSC ESD Coral Bleaching dataset before it is used for stage 2. Run in Google Colab; no GPU is needed.

### Setup: libraries

In [ ]:
!pip install -q datasets

import datasets
import huggingface_hub

print("datasets version: {}".format(datasets.__version__))
print("huggingface_hub version: {}".format(huggingface_hub.__version__))

## Section 1: Do the labels load?

Question: does `load_dataset` return the two labels (CORAL and CORAL_BL) for every split, with the counts the folders hold?

Checks: count the image files in each split and class folder of the repository, load the dataset with the default settings, then compare the row counts with the folder counts and look for a label column.

### Files in the repository

In [ ]:
from collections import Counter
from huggingface_hub import list_repo_files

repo_id = "NMFS-OSI/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
image_endings = (".jpg", ".jpeg", ".png")

files = list_repo_files(repo_id, repo_type="dataset")
image_files = [name for name in files if name.lower().endswith(image_endings)]
other_files = [name for name in files if not name.lower().endswith(image_endings)]

print("Files in the repository: {}".format(len(files)))
print("Image files: {}".format(len(image_files)))
print("Other files: {}".format(other_files[:20]))

folder_counts = Counter()
for name in image_files:
    folder_counts["/".join(name.split("/")[:-1])] += 1

for folder in sorted(folder_counts):
    print("{}: {} images".format(folder, folder_counts[folder]))

### Default load

In [ ]:
import os
import time
from datasets import load_dataset

start = time.time()
noaa = load_dataset(repo_id)
print("Loading took {:.1f} seconds".format(time.time() - start))
print(noaa)

for split in noaa:
    print("{}: {} rows, features: {}".format(split, noaa[split].num_rows, noaa[split].features))

cache_bytes = 0
for root, dirs, names in os.walk(os.path.expanduser("~/.cache/huggingface")):
    for name in names:
        path = os.path.join(root, name)
        if not os.path.islink(path):
            cache_bytes += os.path.getsize(path)
print("Hugging Face cache on the Colab disk: {:.2f} GB".format(cache_bytes / 1024 ** 3))

### Row counts against the folders

In [ ]:
top_folders = sorted(set(folder.split("/")[0] for folder in folder_counts))
print("Top folders in the repository: {}".format(top_folders))
print("Splits returned by load_dataset: {}".format(list(noaa.keys())))

for split in noaa:
    folder_total = sum(count for folder, count in folder_counts.items() if folder.split("/")[0] == split)
    has_label = "label" in noaa[split].column_names
    print("{}: {} rows loaded, {} image files under the folder '{}'".format(
        split, noaa[split].num_rows, folder_total, split))
    print("{} has a label column: {}".format(split, has_label))
    if has_label:
        print("{} label counts: {}".format(split, dict(Counter(noaa[split]["label"]))))

**Observation**: